### VFX Movies  Neo4j Graph Pipeline
**Dataset:** TMDB_all_movies.csv  
**Goal:** Build a Neo4j graph of VFX movies & directors, then run 3 business-oriented Cypher queries.


In [1]:
import duckdb
import pandas as pd

df = pd.read_csv("TMDB_all_movies.csv")
print("Columns:", df.columns.tolist())  # sanity check
df.head()

Columns: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'budget', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'cast', 'director', 'director_of_photography', 'writers', 'producers', 'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path']


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.106,371.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,...,suomi,"Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9740.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.300,435.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,...,"svenska, suomi, English","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8622.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.900,2819.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,English,"Sammi Davis, Marc Lawrence, Alicia Witt, Madon...","Robert Rodriguez, Allison Anders, Quentin Tara...","Rodrigo García, Guillermo Navarro, Phil Parmet...","Allison Anders, Robert Rodriguez, Alexandre Ro...","Quentin Tarantino, Alexandre Rockwell, Lawrenc...",Combustible Edison,6.7,116860.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,6,Judgment Night,6.500,370.0,Released,1993-10-15,12136938.0,109.0,21000000.0,tt0107286,...,English,"Jeremy Piven, Lydell M. Cheshier, Michael DeLo...",Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Lloyd Segan, Gene Levy, Marilyn Vance",Alan Silvestri,6.6,21036.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,8,Life in Loops (A Megacities RMX),7.200,30.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,Wolfgang Thaler,"Timo Novotny, Michael Glawogger","Timo Novotny, Ulrich Gehmacher",NaN,8.1,285.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg


#### 1. Loading Raw Data into DuckDB

In [2]:
FILE_PATH = "TMDB_all_movies.csv"

conn = duckdb.connect(FILE_PATH)

# Load CSV into raw table
conn.sql(f"""
    CREATE OR REPLACE TABLE raw_movies AS
    SELECT * FROM read_csv_auto('{FILE_PATH}',
        nullstr=['', 'NA', 'N/A'],
        header=true
    )
""")

# Inspect
conn.sql("DESCRIBE raw_movies").show()
conn.sql("SELECT COUNT(*) AS total FROM raw_movies").show()
conn.sql("SELECT * FROM raw_movies LIMIT 5").show()

┌─────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │ column_type │  null   │   key   │ default │  extra  │
│         varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ title                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ vote_average            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ vote_count              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ status                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ release_date            │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ runtime                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ budget        

#### 2. Data Cleaning

In [4]:
conn.sql("""
    CREATE OR REPLACE TABLE clean_movies AS
    SELECT
        id,
        title,
        original_title,
        TRIM(director)                                       AS director,
        TRY_CAST(release_date AS DATE)                       AS release_date,
        YEAR(TRY_CAST(release_date AS DATE))                 AS release_year,
        genres,
        overview,
        tagline,
        original_language,
        production_companies,
        production_countries,
        spoken_languages,
        "cast"                                               AS cast_list,
        writers,
        producers,
        music_composer,
        director_of_photography,
        popularity,

        -- Treat 0 as missing for budget and revenue
        NULLIF(budget, 0)                                    AS budget,
        NULLIF(revenue, 0)                                   AS revenue,

        -- ROI only when both are valid
        CASE
            WHEN budget > 0 AND revenue > 0
            THEN ROUND((revenue - budget) / budget * 100, 2)
        END                                                  AS roi_pct,

        runtime,
        ROUND(imdb_rating, 1)                                AS rating,
        NULLIF(imdb_votes, 0)                                AS vote_count,
        vote_average,
        status,
        poster_path

    FROM raw_movies
    WHERE title      IS NOT NULL
      AND status     = 'Released'
      AND director   IS NOT NULL
      AND TRIM(director) != ''
""")

# Verify — null_director should be 0
conn.sql("""
    SELECT
        COUNT(*)                   AS total_rows,
        COUNT(budget)              AS has_budget,
        COUNT(revenue)             AS has_revenue,
        COUNT(rating)              AS has_rating,
        COUNT(director)            AS has_director,
        COUNT(*) - COUNT(director) AS null_director
    FROM clean_movies
""").show()

┌────────────┬────────────┬─────────────┬────────────┬──────────────┬───────────────┐
│ total_rows │ has_budget │ has_revenue │ has_rating │ has_director │ null_director │
│   int64    │   int64    │    int64    │   int64    │    int64     │     int64     │
├────────────┼────────────┼─────────────┼────────────┼──────────────┼───────────────┤
│     977136 │      70939 │       25835 │     440788 │       977136 │             0 │
└────────────┴────────────┴─────────────┴────────────┴──────────────┴───────────────┘



#### 3. Filtering VFX Movies
> Since the CSV has no `keywords` column, VFX detection uses `genres`, `overview`, and `tagline`.

In [5]:
conn.sql("""
    CREATE OR REPLACE TABLE vfx_movies AS
    SELECT *
    FROM clean_movies
    WHERE (
        -- Genre-based
        LOWER(genres) LIKE '%science fiction%'
        OR LOWER(genres) LIKE '%action%'
        OR LOWER(genres) LIKE '%fantasy%'
        OR LOWER(genres) LIKE '%adventure%'
        OR LOWER(genres) LIKE '%animation%'
        -- Overview / tagline keyword signals (replaces missing keywords column)
        OR LOWER(overview) LIKE '%visual effects%'
        OR LOWER(overview) LIKE '%cgi%'
        OR LOWER(overview) LIKE '%special effects%'
        OR LOWER(overview) LIKE '%computer generated%'
        OR LOWER(tagline)  LIKE '%visual effects%'
        OR LOWER(tagline)  LIKE '%cgi%'
        OR LOWER(tagline)  LIKE '%special effects%'
    )
""")

conn.sql("SELECT COUNT(*) AS vfx_movie_count FROM vfx_movies").show()
conn.sql("SELECT title, genres, release_year FROM vfx_movies LIMIT 10").show()

┌─────────────────┐
│ vfx_movie_count │
│      int64      │
├─────────────────┤
│          155439 │
└─────────────────┘

┌────────────────────────────────────────────────────────┬────────────────────────────────────┬──────────────┐
│                         title                          │               genres               │ release_year │
│                        varchar                         │              varchar               │    int64     │
├────────────────────────────────────────────────────────┼────────────────────────────────────┼──────────────┤
│ Judgment Night                                         │ Action, Crime, Thriller            │         1993 │
│ Star Wars                                              │ Adventure, Action, Science Fiction │         1977 │
│ Finding Nemo                                           │ Animation, Family, Adventure       │         2003 │
│ The Fifth Element                                      │ Science Fiction, Action, Adventure │       

#### 4. Exporting Parquet Snapshot

In [7]:
conn.sql("""
    COPY vfx_movies
    TO 'vfx_movies_1.parquet'
    (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

print("VFX Parquet saved!")

VFX Parquet saved!


#### 5. Exporting Neo4j CSVs
Three files:
- `neo4j_movies_1.csv` — Movie nodes
- `neo4j_directors_1.csv` — Director nodes
- `neo4j_directed_by_1.csv` — DIRECTED relationships

In [11]:
OUTPUT_DIR = "."

# Movie nodes
conn.sql(f"""
    COPY (
        SELECT DISTINCT
            id           AS movieId,
            title,
            release_year,
            rating,
            vote_average,
            budget,
            revenue,
            roi_pct,
            runtime,
            genres,
            original_language
        FROM vfx_movies
        WHERE title IS NOT NULL
    )
    TO '{OUTPUT_DIR}/neo4j_movies_1.csv' (HEADER, DELIMITER ',')
""")

# Director nodes
conn.sql(f"""
    COPY (
        SELECT DISTINCT
            director AS name
        FROM vfx_movies
        WHERE director IS NOT NULL
          AND TRIM(director) != ''
    )
    TO '{OUTPUT_DIR}/neo4j_directors_1.csv' (HEADER, DELIMITER ',')
""")

# DIRECTED relationships
conn.sql(f"""
    COPY (
        SELECT DISTINCT
            id       AS movieId,
            director AS directorName
        FROM vfx_movies
        WHERE director IS NOT NULL
          AND TRIM(director) != ''
    )
    TO '{OUTPUT_DIR}/neo4j_directed_by_1.csv' (HEADER, DELIMITER ',')
""")

print("Neo4j VFX CSVs ready!")

Neo4j VFX CSVs ready!


#### 6. Director Null Audit (Verification)

In [12]:
conn.sql("""
    SELECT 
        COUNT(*)                                      AS total_movies,
        COUNT(director)                               AS has_director,
        COUNT(*) - COUNT(director)                    AS null_director,
        ROUND(100.0 * COUNT(director) / COUNT(*), 1)  AS pct_with_director
    FROM clean_movies
""").show()

┌──────────────┬──────────────┬───────────────┬───────────────────┐
│ total_movies │ has_director │ null_director │ pct_with_director │
│    int64     │    int64     │     int64     │      double       │
├──────────────┼──────────────┼───────────────┼───────────────────┤
│       977136 │       977136 │             0 │             100.0 │
└──────────────┴──────────────┴───────────────┴───────────────────┘



---
#### 9. Running Cypher Queries from Python (neo4j driver)

In [ ]:

from neo4j import GraphDatabase
import pandas as pd

URI      = "Your URI"
USERNAME = "neo4j"
PASSWORD = "Your Password"   # <-- change to your Neo4j password

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def run_query(cypher, label="Result"):
    with driver.session() as session:
        result = session.run(cypher)
        df = pd.DataFrame([r.data() for r in result])
    print(f"\n=== {label} ===")
    print(df.to_string(index=False))
    return df

In [14]:
# Query 1 — Network Reach
q1 = """
MATCH (d:Director)-[:DIRECTED]->(m:Movie)
WHERE m.year > 2010
RETURN d.name AS director, COUNT(m) AS recent_films
ORDER BY recent_films DESC
LIMIT 10
"""
df_q1 = run_query(q1, "Query 1 — Network Reach (Most Prolific VFX Directors since 2010)")


=== Query 1 — Network Reach (Most Prolific VFX Directors since 2010) ===
        director  recent_films
      Kevin Dunn           113
 Koichi Sakamoto            65
  Hiroyuki Tsuji            63
      Ken Jacobs            57
   Vince McMahon            51
     Julia Ocker            49
Petter Baiestorf            41
     Evan Tramel            40
   Devon Damonte            33
           YAGOO            32


In [15]:
# Query 2 — Grouped Comparison
q2 = """
MATCH (d:Director)-[:DIRECTED]->(m:Movie)
WHERE m.roi_pct IS NOT NULL
WITH d.name AS director, AVG(m.roi_pct) AS avg_roi, COUNT(m) AS film_count
WHERE film_count >= 2
RETURN director, ROUND(avg_roi, 1) AS avg_roi_pct, film_count
ORDER BY avg_roi_pct DESC
LIMIT 10
"""
df_q2 = run_query(q2, "Query 2 — Grouped Comparison (Highest Avg ROI Directors)")


=== Query 2 — Grouped Comparison (Highest Avg ROI Directors) ===
        director  avg_roi_pct  film_count
     Itamar kima  266666616.7           2
    Eder Rigolon     100000.0           2
Augusto Salvador      54900.0           2
         Guan Hu      28026.7           2
     Mister Hero      15650.0           2
 Jose N. Carreon       9900.0           2
 Edgardo Vinarao       9900.0           2
  Haruo Sotozaki       9625.7           3
   Dennis Hopper       7788.5           2
George A. Romero       5872.7           6


In [16]:
# Query 3 — Collaboration / Affinity
q3 = """
MATCH (a:Director)-[:DIRECTED]->(m1:Movie),
      (b:Director)-[:DIRECTED]->(m2:Movie)
WHERE id(a) < id(b)
  AND m1.genres IS NOT NULL
  AND m1.genres = m2.genres
WITH a, b, COUNT(*) AS shared_genre_combos
WHERE shared_genre_combos > 3
RETURN a.name AS director_a, b.name AS director_b, shared_genre_combos
ORDER BY shared_genre_combos DESC
LIMIT 10
"""
df_q3 = run_query(q3, "Query 3 — Affinity (Directors Working in Same Genre Space)")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=7, offset=94>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 94, 'line': 4, 'column': 7}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (a:Director)-[:DIRECTED]->(m1:Movie),\n      (b:Director)-[:DIRECTED]->(m2:Movie)\nWHERE id(a) < id(b)\n  AND m1.genres IS NOT NULL\n  AND m1.genres = m2.genres\nWITH a, b, COUNT(*) AS shared_genre_combos\nWHERE shared_genre_combos > 3\nRETURN a.name AS director_a, b.name AS director_b, shared_genre_combos\


=== Query 3 — Affinity (Directors Working in Same Genre Space) ===
     director_a     director_b  shared_genre_combos
     Paul Terry Dave Fleischer                80469
Seymour Kneitel Dave Fleischer                67354
  Vince McMahon     Kevin Dunn                45356
   Otto Messmer Dave Fleischer                45210
Connie Rasinski Dave Fleischer                44425
Seymour Kneitel     Paul Terry                40993
   Otto Messmer     Paul Terry                39500
 Dave Fleischer   Friz Freleng                38717
   Walter Lantz Dave Fleischer                37679
   Mannie Davis Dave Fleischer                36443


In [18]:
driver.close()
print("Done. Three queries executed.")

Done. Three queries executed.
